# Importação das bibliotecas necessárias

In [1]:
import sys
import os
sys.path.append('/home/jovyan/work')

from utils.AlinharETL import AlinharETL

# Crie uma instância da classe AlinharETL

In [2]:
# Crie uma instância da classe
bucket = "gold"
datamart = "sgt"
extrator_bronze = AlinharETL(bucket,datamart)

2024-09-25 17:09:05,754 - INFO - Iniciando Sessão Spark.


# Leitura da tabela

In [3]:
df_relatorios_sgt = extrator_bronze.criar_view_temporaria('silver/SGT/RelatoriosSgt','sgt_relatorios')

2024-09-25 17:09:10,753 - INFO - Aguarde... Criando View (silver/SGT/RelatoriosSgt)
2024-09-25 17:09:22,312 - INFO - View criada com sucesso


In [4]:
df_clientes_sgt = extrator_bronze.criar_view_temporaria('silver/SGT/ClientesSgt','sgt_clientes')

2024-09-25 17:09:22,318 - INFO - Aguarde... Criando View (silver/SGT/ClientesSgt)
2024-09-25 17:09:22,925 - INFO - View criada com sucesso


In [5]:
df_atendimentos_sgt = extrator_bronze.criar_view_temporaria('silver/SGT/AtendimentoSgt','sgt_atendimentos')

2024-09-25 17:09:22,931 - INFO - Aguarde... Criando View (silver/SGT/AtendimentoSgt)
2024-09-25 17:09:23,513 - INFO - View criada com sucesso


# Tratamentos na tabela 

In [6]:
sql_query = """

SELECT cpf_cnpj, nome_cliente
FROM (
    SELECT 
        cpf_cnpj, 
        nome_cliente, 
        ROW_NUMBER() OVER(PARTITION BY cpf_cnpj ORDER BY cpf_cnpj) as row_num
    FROM (
        SELECT nr_cnpj_cliente AS cpf_cnpj, nm_cliente AS nome_cliente
        FROM sgt_relatorios

        UNION ALL

        SELECT nr_cpf_cnpj_cliente AS cpf_cnpj, nm_cliente AS nome_cliente
        FROM sgt_atendimentos

        UNION ALL

        SELECT nr_cpf_cnpj AS cpf_cnpj, dsc_nome AS nome_cliente
        FROM sgt_clientes
    ) as combined_data
) as numbered_rows
WHERE row_num = 1 and cpf_cnpj IS NOT NULL;

"""

In [7]:
sgt_empresas = extrator_bronze.carregar_dados_delta(sql_query)

2024-09-25 17:09:23,556 - INFO - Aguarde... Executando Query (Delta)
2024-09-25 17:09:23,558 - INFO - 

SELECT cpf_cnpj, nome_cliente
FROM (
    SELECT 
        cpf_cnpj, 
        nome_cliente, 
        ROW_NUMBER() OVER(PARTITION BY cpf_cnpj ORDER BY cpf_cnpj) as row_num
    FROM (
        SELECT nr_cnpj_cliente AS cpf_cnpj, nm_cliente AS nome_cliente
        FROM sgt_relatorios

        UNION ALL

        SELECT nr_cpf_cnpj_cliente AS cpf_cnpj, nm_cliente AS nome_cliente
        FROM sgt_atendimentos

        UNION ALL

        SELECT nr_cpf_cnpj AS cpf_cnpj, dsc_nome AS nome_cliente
        FROM sgt_clientes
    ) as combined_data
) as numbered_rows
WHERE row_num = 1


2024-09-25 17:09:23,784 - INFO - Query (Delta) executada com sucesso


# Gravação no datalake

In [8]:
extrator_bronze.caminho_tabela_delta = 's3a://gold/SGT/DimEmpresasSgt'

In [9]:
extrator_bronze.salvar_delta(sgt_empresas, 'overwrite')

2024-09-25 17:09:23,798 - INFO - Aguarde... Persistindo dados (overwrite)
2024-09-25 17:09:39,745 - INFO - Dados persistidos com sucesso
2024-09-25 17:09:39,746 - INFO - s3a://gold/SGT/DimEmpresasSgt
2024-09-25 17:09:39,746 - INFO - Aguarde... Realizando optimize
2024-09-25 17:09:41,948 - INFO - Optimize executado com sucesso.
2024-09-25 17:09:41,949 - INFO - Aguarde... Realizando vacuum
2024-09-25 17:10:08,750 - INFO - Vacuum executado com sucesso.


# Encerra sessão spark

In [10]:
extrator_bronze.parar_sessao()

2024-09-25 17:10:09,036 - INFO - Sessão Spark finalizada.
